In [166]:
def multivariable_division(f, G, R):
    x = R.gen()
    
    p = R(f)
    G = [R(g) for g in G]
    Q = [R(0) for q in G]
    r = R(0)

    while p != 0:
        p = R(p)
        lc_p = p.leading_coefficient()
        lm_p = p.leading_monomial()
        
        division_occoured = False
        i = 0
        
        while i < len(G) and not division_occoured:
            
            gi = G[i]

            lc_g = gi.leading_coefficient()
            lm_g = gi.leading_monomial()
            
            if lm_g.divides(lm_p):
                
                m = (lc_p/lc_g) * (lm_p / lm_g)
                Q[i] += m
                p -= m * gi

                division_occoured = True
            else:
                i += 1
                
        if not division_occoured:
            r += p.leading_term()
            p -= p.leading_term()

    return Q, r

In [3]:
def S_polynomial(f, g):
    f = R(f)
    g = R(g)
    LCf = f.leading_coefficient()
    LCg = g.leading_coefficient()
    LMf = f.leading_monomial()
    LMg = g.leading_monomial()
    l = lcm(LMf, LMg)
    
    s = (l/LMf)* 1/LCf * f - (l/LMg) * 1/LCg * g
    return R(s)

In [131]:
def buchberger_alg_s(F, R):
    G = [R(f) for f in F]
    counts = {
        "pairs_total": 0,
        "pairs_skipped": 0,
        "s_computed": 0,
    }

    while True:
        Gprime = list(G)
        
        for i in range(len(Gprime)):
            for j in range(i+1, len(Gprime)):
                counts["pairs_total"] += 1
                p = Gprime[i]
                q = Gprime[j]

                LMp = p.leading_monomial()
                LMq = q.leading_monomial()

                if lcm(LMp, LMq) == LMp * LMq:
                    counts["pairs_skipped"] += 1
                    continue
                    
                counts["s_computed"] += 1
                S = S_polynomial(p, q)
                Q, r = multivariable_division(S, Gprime, R)
                    
                if r != 0:
                    if r.is_constant():
                        return [R(1)]
                    r = r / r.leading_coefficient()
                    if r not in G:
                        G.append(r)

        if G == Gprime:
            return G, counts

In [161]:
R.<x,y,z> = PolynomialRing(QQ, order='degrevlex')

f1 = x^2 + y^2 + z^2
f2 = x*y + x*z + y*z
f3 = x*y*z
F = [f1, f2, f3]
%timeit G, counts = buchberger_alg_s(F, R)
print(G, len(G), counts)

152 ms ± 411 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)
[x^2 + y^2 + z^2, x*y + x*z + y*z, x*y*z, y^3 + y^2*z + x*z^2 + 2*y*z^2 + z^3, y^3*z + y*z^3, x*z^2 + y*z^2, y^2*z^2 + z^4, y^2*z^3 + 1/2*y*z^4 + 1/2*z^5, y^2*z^2, y^2*z^4 - z^6, y*z^4 - z^5, z^4, y*z^5 - 2*z^6, z^5, y*z^6 + z^7, y*z^4, y*z^5, y*z^6, y*z^5 + z^6, z^6, y*z^4 + z^5, y*z^5 + 3*z^6] 22 {'pairs_total': 294, 'pairs_skipped': 35, 's_computed': 259}


In [171]:
def buchberger_alg(F, R):
    G = [R(f) for f in F]
    scount = 0
    while True:
        Gprime = list(G)
        
        for i in range(len(Gprime)):
            for j in range(i+1, len(Gprime)):
                p = Gprime[i]
                q = Gprime[j]
                scount += 1
                S = S_polynomial(p, q)
                Q, r = multivariable_division(S, Gprime, R)

                if r != 0:
                    if r.is_constant():
                        return [R(1)]
                    r = r / r.leading_coefficient()
                    if r not in G:
                        G.append(r)

        if G == Gprime:
            return G, scount

In [190]:
R.<x,y,z> = PolynomialRing(QQ, order='lex')

f1 = 2*x^2 + 3*y^3 + 5*z^2
f2 = 7*x^2*y + 11*x*z + 13*y*z
F = [f1, f2] # Generators of the ideal

G, scount = buchberger_alg(F, R)
m = max(g.total_degree() for g in G)
print(G, len(G), scount, m)

[2*x^2 + 3*y^3 + 5*z^2, 7*x^2*y + 11*x*z + 13*y*z, x*z - 21/22*y^4 - 35/22*y*z^2 + 13/11*y*z, x*y^4 + 35/22*y^5*z - 13/11*y^5 + 11/7*y^3*z + 175/66*y^2*z^3 - 130/33*y^2*z^2 + 338/231*y^2*z + 55/21*z^3, x*y^5 + 35/22*y^6*z - 13/11*y^6 + 11/7*y^4*z + 175/66*y^3*z^3 - 130/33*y^3*z^2 + 338/231*y^3*z + 55/21*y*z^3, y^9 + 10/3*y^6*z^2 - 52/21*y^6*z + 242/147*y^4*z^2 + 25/9*y^3*z^4 - 260/63*y^3*z^3 + 676/441*y^3*z^2 + 1210/441*y*z^4, y^10 + 10/3*y^7*z^2 - 52/21*y^7*z + 242/147*y^5*z^2 + 25/9*y^4*z^4 - 260/63*y^4*z^3 + 676/441*y^4*z^2 + 1210/441*y^2*z^4, y^8 + 10/3*y^5*z^2 - 52/21*y^5*z + 242/147*y^3*z^2 + 25/9*y^2*z^4 - 260/63*y^2*z^3 + 676/441*y^2*z^2 + 1210/441*z^4] 8 42 10


In [169]:
R.<x,y> = PolynomialRing(QQ, order='lex')

f1 = 3*x^3*y^2 + 4*x*y^2 # Generators of the ideal
f2 = y^3 + xy
F = [f1, f2]

G = buchberger_alg(F, R) # Construct Grobner basis

f = x*y^2 - x # Test polynomial

Q, r = multivariable_division(f, G, R) # Compute remainder

print('G =', G,', Q =', Q, ' with Remainder ', r)
\end{lstlisting}

G = [3*x^3*y^2 + 4*x*y^2, x*y + y^3, 2*x^2 - y^5, y^8 + 4/3*y^4, y^9 + 8/3*y^4, y^6 - 2*y^5, y^5 - 2*y^4, y^5 + 1/6*y^4, y^4] , Q = [0, y, 0, 0, 0, 0, 0, 0, -1]  with Remainder  -x
